In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install py_vncorenlp underthesea pyserini faiss-cpu

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 6.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.2/412.2 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.7/196.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os

def install_java():
  !apt update -qq
  !apt-get install -y openjdk-21-jdk-headless -qq > /dev/null
  os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
  !java -version

install_java()

6 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "21.0.9" 2025-10-21
OpenJDK Runtime Environment (build 21.0.9+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.9+10-Ubuntu-122.04, mixed mode, sharing)


In [ ]:
!git clone https://github.com/Tommachilez/fact-checking-data-framework-vn.git
%cd /content/fact-checking-data-framework-vn/

Cloning into 'fact-checking-data-framework-vn'...
remote: Enumerating objects: 580, done.
remote: Counting objects: 100% (236/236), done.
remote: Compressing objects: 100% (169/169), done.
remote: Total 580 (delta 160), reused 122 (delta 67), pack-reused 344 (from 1)
Receiving objects: 100% (580/580), 5.37 MiB | 4.08 MiB/s, done.
Resolving deltas: 100% (384/384), done.
/content/fact-checking-data-framework-vn


In [ ]:
VNCORE_MODEL_PATH = "/content/drive/MyDrive/KLTN_Project/Models/vncorenlp_models"
dataset_path = "/content/drive/MyDrive/KLTN_Project/Datasets"
csv_path = f"{dataset_path}/vifc.csv"
jsonl_path = f"{dataset_path}/vifc_queries_triples.jsonl"
stopwords_path = f"{dataset_path}/stopwords-vi.txt"
output_dir = f"{dataset_path}/filtered_queries"

# **Filter**

In [ ]:
!python -m src.scripts.filtering.query_lexical_filter \
    --vncorenlp {VNCORE_MODEL_PATH} \
    --csv {csv_path} \
    --jsonl {jsonl_path} \
    --stopwords {stopwords_path} \
    --output_dir {output_dir} \
    --threshold 0.5 \
    # --quota 10

2025-12-26 13:19:33 INFO  WordSegmenter:24 - Loading Word Segmentation model
Loaded 645 stopwords (Whitelist enabled: False).
Loaded 34811 documents.
Output directory set to: /content/drive/MyDrive/KLTN_Project/Datasets/filtered_queries
Counting lines in JSONL file...
Processing 34807 entries with threshold 0.5...
Filtering:   0% 0/34807 [00:00<?, ?docs/s]
[DEBUG] First Entry ID in JSON: '0'
[DEBUG] ID '0' FOUND in CSV documents.
[DEBUG] Queries structure: [{"query": "Phó Thủ tướng Trần Hồng Hà chúc mừng Đài Truyền hình Việt Nam Hải Phòng COVID-19", "type": "KEYWORD"}, {"query": "Phó Thủ tướng Trần Hồng Hà đã chúc mừng những đơn vị nào sau 2 năm gián đo...
[DEBUG] Type: keyword | Overlap: 1.00 | Threshold: 0.5
[DEBUG] Type: natural | Overlap: 1.00 | Threshold: 0.5
Filtering:   0% 1/34807 [00:01<15:21:59,  1.59s/docs][DEBUG] Type: keyword | Overlap: 1.00 | Threshold: 0.5
[DEBUG] Type: natural | Overlap: 0.90 | Threshold: 0.5
Filtering:   0% 2/34807 [00:01<7:30:40,  1.29docs/s] [DEBUG] T

# **BM25 Hard Negative Mining**

In [ ]:
filter_path = f"{output_dir}/filtered_source.jsonl"
mining_dir = f"{dataset_path}/vn_mining"
training_triples = f"{dataset_path}/training_triples.jsonl"
doc_mapping = f"{dataset_path}/unique_doc_mapping.csv"
train_csv = f"{dataset_path}/vifc_train.csv"

In [ ]:
# Step 1: Preprocessing (Uses PyVnCoreNLP)
!python -m src.scripts.hard_negative_mining.preprocess \
    --full_data_csv {csv_path} \
    --doc_mapping {doc_mapping} \
    --query_jsonl {filter_path} \
    --train_csv {train_csv} \
    --vncorenlp_path {VNCORE_MODEL_PATH} \
    --stopwords_path {stopwords_path} \
    --output_dir {mining_dir} \
    --claim_col "query"

2025-12-26 14:40:25 INFO  WordSegmenter:24 - Loading Word Segmentation model
>>> Loading Document Mapping (Canonical IDs)...
>>> Processing Documents from Mapping...
Tokenizing Corpus: 3030it [00:47, 63.38it/s]
>>> Loading Full Data CSV (Original IDs)...
>>> Processing Queries & Resolving IDs...
   > Processing Generated Queries from /content/drive/MyDrive/KLTN_Project/Datasets/filtered_queries/filtered_source.jsonl
Generated Queries: 34807it [01:16, 456.92it/s]
   > Processing Claims from /content/drive/MyDrive/KLTN_Project/Datasets/vifc_train.csv
Train Claims: 27124it [00:34, 779.72it/s]
>>> Preprocessing complete.
    Files saved to /content/drive/MyDrive/KLTN_Project/Datasets/vn_mining
    Queries Resolved: 61931
    Queries Dropped (Text mismatch between Full Data and Mapping): 0


In [ ]:
# Step 2: Mining (Uses Pyserini)
# Note: This is a separate process execution, so a new JVM is started.
!python -m src.scripts.hard_negative_mining.pyserini_mining \
    --preprocessed_dir {mining_dir} \
    --doc_mapping {doc_mapping} \
    --output_jsonl {training_triples} \
    --top_k 10

2025-12-26 14:43:26.913828: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-26 14:43:27.385183: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-26 14:43:27.482329: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766760207.538181    3638 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766760207.555590    3638 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766760207.600277    3638 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [ ]:
import json
import os

class DatasetViewer:
    def __init__(self, corpus_path: str, queries_path: str, triples_path: str):
        """
        Initialize with paths to the three key files.
        """
        self.paths = {
            "corpus": corpus_path,
            "queries": queries_path,
            "triples": triples_path
        }

    def _view_file(self, key: str, n: int):
        path = self.paths.get(key)
        title = key.replace('_', ' ').title()

        print(f"\n{'='*20} VIEWING: {title} (First {n} lines) {'='*20}")

        if not path or not os.path.exists(path):
            print(f"❌ File not found at: {path}")
            return

        try:
            with open(path, 'r', encoding='utf-8') as f:
                for i in range(n):
                    line = f.readline()
                    if not line:
                        print(f"--- End of file reached at line {i} ---")
                        break

                    try:
                        # Parse JSON to pretty-print it
                        data = json.loads(line.strip())
                        print(f"🔹 [Line {i+1}]")
                        print(json.dumps(data, indent=2, ensure_ascii=False))
                    except json.JSONDecodeError:
                        print(f"⚠️ [Line {i+1}] (Invalid JSON)")
                        print(line.strip())
        except Exception as e:
            print(f"Error reading file: {e}")

    def view_corpus(self, n: int = 5):
        """View the pretokenized corpus file."""
        self._view_file("corpus", n)

    def view_queries(self, n: int = 5):
        """View the pretokenized queries file."""
        self._view_file("queries", n)

    def view_triples(self, n: int = 5):
        """View the final output training triples."""
        self._view_file("triples", n)

    def view_all(self, n: int = 3):
        """View first n lines of all files sequentially."""
        self.view_corpus(n)
        self.view_queries(n)
        self.view_triples(n)

In [ ]:
viewer = DatasetViewer(
    f"{mining_dir}/corpus_pretokenized.jsonl",
    f"{mining_dir}/queries_pretokenized.jsonl",
    training_triples,
)

In [ ]:
viewer.view_all()


==================== VIEWING: Corpus (First 3 lines) ====================
🔹 [Line 1]
{
  "id": "0",
  "contents": "chinhphu.vn mong_muốn gửi_gắm phó thủ_tướng trần hồng hà truyền_hình lễ bế_mạc liên_hoan truyền_hình toàn_quốc thứ 41 tối 18/3 tp hải_phòng phó thủ_tướng trần hồng hà tác_phẩm truyền_hình vun_đắp làm_giàu văn_hoá việt_nam tiên_tiến đậm_đà bản_sắc dân_tộc góp_phần tạo_dựng môi_trường văn_hoá lành_mạnh xây_dựng con_người việt_nam nhân_cách trách_nhiệm hội_nhập ảnh vgp minh khôi tham_dự lễ bế_mạc bí_thư trung_ương đảng trưởng ban tuyên_giáo trung_ương nguyễn_trọng nghĩa lãnh_đạo ngành trung_ương địa_phương đại_diện đài_truyền_hình đơn_vị sản_xuất chương_trình truyền_hình đông_đảo cán_bộ phóng_viên biên_tập_viên nghệ_sĩ diễn_viên hoạt_động lĩnh_vực truyền_hình … thay_mặt chính_phủ thủ_tướng chính_phủ phó thủ_tướng trần hồng hà chúc_mừng đài_truyền_hình việt_nam đài_truyền_hình tỉnh thành_phố nước đơn_vị sản_xuất truyền_hình tp hải_phòng tổ_chức thành_công sự_kiện quan_trọng 2

In [3]:
import json
import os

def extract_queries(file_path):
    """
    Extracts a set of query strings from a JSONL file.
    Handles 'generated_queries' (primary) and 'queries' (fallback) keys.
    Returns a set of unique queries and a list of all queries found.
    """
    unique_queries = set()
    all_queries = []
    total_docs = 0

    print(f"📂 Processing: {os.path.basename(file_path)}...")

    if not os.path.exists(file_path):
        print(f"   ❌ File not found: {file_path}")
        return set(), []

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue

                try:
                    data = json.loads(line)
                    total_docs += 1

                    # Determine which key holds the list of queries
                    query_list = data.get("generated_queries", data.get("queries", []))

                    # If it's a simple triple file (one query per line), wrap it
                    if "query" in data and not query_list:
                        if isinstance(data["query"], str):
                            query_list = [{"query": data["query"]}]

                    # Extract query text
                    for item in query_list:
                        text = ""
                        if isinstance(item, dict):
                            text = item.get("query", "").strip()
                        elif isinstance(item, str):
                            text = item.strip()

                        if text:
                            unique_queries.add(text)
                            all_queries.append(text)

                except json.JSONDecodeError:
                    continue

    except Exception as e:
        print(f"   ⚠️ Error reading file: {e}")

    print(f"   ✅ Documents: {total_docs}")
    print(f"   ✅ Total Queries found: {len(all_queries)}")
    print(f"   ✅ Unique Queries: {len(unique_queries)}")
    print("-" * 40)
    return unique_queries, all_queries

def analyze_duplicates(file_a, file_b):
    """
    Analyzes duplicates between two files and internal duplicates.
    """
    print(f"--- 🔍 STARTING ANALYSIS ---")

    # Extract
    unique_a, all_a = extract_queries(file_a)
    unique_b, all_b = extract_queries(file_b)

    # 1. Internal Duplicates (Repetition within files)
    internal_dupes_a = len(all_a) - len(unique_a)
    internal_dupes_b = len(all_b) - len(unique_b)

    print(f"📊 Internal Duplication Analysis:")
    print(f"   - {os.path.basename(file_a)} duplicates: {internal_dupes_a}")
    print(f"   - {os.path.basename(file_b)} duplicates: {internal_dupes_b}")

    # 2. Cross-File Intersection (A vs B)
    intersection = unique_a.intersection(unique_b)
    only_in_a = unique_a - unique_b
    only_in_b = unique_b - unique_a

    print(f"\n🔗 Cross-File Analysis:")
    print(f"   - Common Queries (Intersection): {len(intersection)}")
    print(f"   - Only in Source ({os.path.basename(file_a)}): {len(only_in_a)} (Filtered Out)")
    print(f"   - Only in Filtered ({os.path.basename(file_b)}): {len(only_in_b)} (New/Unexpected?)")

    # Sanity Check
    if len(only_in_b) > 0:
        print("\n⚠️ WARNING: The filtered file contains queries NOT found in the source file.")
        print("   First 5 unexpected queries:", list(only_in_b)[:5])
    else:
        print("\n✅ VALIDATION PASSED: Filtered file is a strict subset of the source.")

# --- Usage ---
dataset_path = "/content/drive/MyDrive/KLTN_Project/Datasets"
file_source = f"{dataset_path}/vifc_queries_triples.jsonl"
file_filtered = f"{dataset_path}/filtered_queries/filtered_source.jsonl"

analyze_duplicates(file_source, file_filtered)

--- 🔍 STARTING ANALYSIS ---
📂 Processing: vifc_queries_triples.jsonl...
   ✅ Documents: 34807
   ✅ Total Queries found: 104583
   ✅ Unique Queries: 102003
----------------------------------------
📂 Processing: filtered_source.jsonl...
   ✅ Documents: 34807
   ✅ Total Queries found: 103797
   ✅ Unique Queries: 101225
----------------------------------------
📊 Internal Duplication Analysis:
   - vifc_queries_triples.jsonl duplicates: 2580
   - filtered_source.jsonl duplicates: 2572

🔗 Cross-File Analysis:
   - Common Queries (Intersection): 101225
   - Only in Source (vifc_queries_triples.jsonl): 778 (Filtered Out)
   - Only in Filtered (filtered_source.jsonl): 0 (New/Unexpected?)

✅ VALIDATION PASSED: Filtered file is a strict subset of the source.
